In [1]:
import re
import numpy as np
import pandas as pd
from pathlib import Path
import lightgbm as lgb
from sklearn.linear_model import RidgeCV
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings("ignore")

# =========================
# CONFIG
# =========================
ROOT     = Path("/home/mohamed/SDD/hackathon/sia-predicting-short-form-video-popularity")
DATA_DIR = ROOT / "Data"

TRAIN_MERGED = DATA_DIR / "X_train_merged.csv"
TEST_MERGED  = DATA_DIR / "X_test_merged.csv"
Y_PATH       = ROOT / "y_train.csv"

N_SPLITS     = 5
SEED         = 42
PCA_CLIP_DIM = 64
PCA_VGG_DIM  = 64
LOG_TARGET   = True

# Colonnes catégorielles à encoder par la target
# (colonnes one-hot up_uploader_* → on reconstitue l'uploader)
TARGET_ENC_COLS = ["uploader_log_count", "channel_log_count", "artist_log_count"]
# + on va créer les target encodings sur les one-hot uploaders reconstitués

# =========================
# COLUMN GROUPS
# =========================
CLIP_COLS = [f"c{i}" for i in range(512)]
VGG_COLS  = [f"vggish_{i:03d}" for i in range(256)]

LIBROSA_COLS = [
    "rms_mean","rms_std","zcr_mean","zcr_std",
    "centroid_mean","centroid_std","bandwidth_mean","bandwidth_std",
    "rolloff_mean","rolloff_std","tempo","onset_rate",
    *[f"mfcc_{i}_{s}" for i in range(6) for s in ("mean","std")],
]

META_COLS = [
    "uploader_log_count","uploader_short_log_count","channel_log_count",
    "artist_log_count","track_log_count","album_log_count","uid_log_count",
    "has_music","is_original_sound","resolution_area","is_vertical",
    "is_full_hd","aspect_w_over_h","nb_words","avg_word_length",
    "has_exclamation","has_question","uppercase_ratio","has_mention",
    "has_url","has_number","words_per_second","uploader_unique_artist",
    "uploader_unique_track","deezer_rank","status",
    "text_len","nb_hashtags","nb_emojis","hashtag_density","emoji_density",
    "sentiment_score","sentiment_abs","nb_textbloc",
    "aspect_ratio","video_duration","release_year",
]

VIDEO_COLS = [
    *[f"f{i}_{s}" for i in range(1,6) for s in ("sharpness","brightness","saturation")],
    "hook_motion","hook_shake","content_motion","content_shake","max_peak_motion",
    "avg_motion","camera_shake","duration","num_cuts",
    "att_mean_saliency","att_max_saliency","att_min_saliency","att_skewness",
    "att_kurtosis","att_hook_score","att_outro_score","att_peak_location",
    "att_high_attention_ratio","att_attention_instability",
    "yolo_person","yolo_ski_snow","avg_face_count","max_face_coverage",
    "body_presence_score",
    "dominant_emotion_angry","dominant_emotion_disgust","dominant_emotion_fear",
    "dominant_emotion_happy","dominant_emotion_neutral","dominant_emotion_none",
    "dominant_emotion_sad","dominant_emotion_surprise",
    *[f"R{i}" for i in range(1,4)], *[f"G{i}" for i in range(1,4)],
    *[f"B{i}" for i in range(1,4)], *[f"W{i}" for i in range(1,4)],
]

ONEHOT_PATTERN = re.compile(r"^(up_|lang_)")

# =========================
# UTILS
# =========================
def read_csv_robust(path):
    try:
        return pd.read_csv(path, sep=None, engine="python")
    except Exception:
        return pd.read_csv(path, sep=",", engine="python")

def normalize_id(s):
    s = s.astype(str).str.strip()
    s = s.str.replace(r"^(VIDEO_|video_|Video_)", "", regex=True)
    s = s.str.extract(r"(\d+)", expand=False).fillna(s)
    return s.str.strip()

def apply_pca(X_tr, X_te, n_comp, name):
    n_comp = min(n_comp, X_tr.shape[1], X_tr.shape[0])
    scaler = StandardScaler()
    Xs_tr  = scaler.fit_transform(np.nan_to_num(X_tr))
    Xs_te  = scaler.transform(np.nan_to_num(X_te))
    pca    = PCA(n_components=n_comp, random_state=SEED)
    tr_out = pca.fit_transform(Xs_tr)
    te_out = pca.transform(Xs_te)
    var    = pca.explained_variance_ratio_.cumsum()[-1]
    print(f"   PCA {name}: {X_tr.shape[1]} → {n_comp} dims | var: {var:.1%}")
    cols   = [f"{name}_pca_{i}" for i in range(n_comp)]
    return pd.DataFrame(tr_out, columns=cols), pd.DataFrame(te_out, columns=cols)

def safe_cols(candidates, df):
    return [c for c in candidates if c in df.columns]

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

# =========================
# TARGET ENCODING (KFold, sans leakage)
# =========================
def target_encode_kfold(train_df, test_df, col, target, n_splits=5, smoothing=10):
    """
    Encode col → moyenne de target, avec :
    - KFold sur train pour éviter le leakage
    - Smoothing bayésien : (n * mean_cat + m * mean_global) / (n + m)
    - Test : moyenne globale par catégorie sur tout le train
    """
    global_mean = target.mean()
    oof_enc     = np.zeros(len(train_df))
    kf          = KFold(n_splits=n_splits, shuffle=True, random_state=SEED)

    for tr_idx, val_idx in kf.split(train_df):
        # Stats calculées sur le fold train uniquement
        fold_stats = (
            pd.DataFrame({"cat": train_df[col].iloc[tr_idx].values, "y": target.iloc[tr_idx].values})
            .groupby("cat")["y"]
            .agg(["mean", "count"])
        )
        # Smoothing bayésien
        fold_stats["smooth"] = (
            (fold_stats["count"] * fold_stats["mean"] + smoothing * global_mean)
            / (fold_stats["count"] + smoothing)
        )
        # Appliquer au fold val
        val_cats = train_df[col].iloc[val_idx].values
        oof_enc[val_idx] = pd.Series(val_cats).map(fold_stats["smooth"]).fillna(global_mean).values

    # Pour le test : stats sur tout le train
    full_stats = (
        pd.DataFrame({"cat": train_df[col].values, "y": target.values})
        .groupby("cat")["y"]
        .agg(["mean", "count"])
    )
    full_stats["smooth"] = (
        (full_stats["count"] * full_stats["mean"] + smoothing * global_mean)
        / (full_stats["count"] + smoothing)
    )
    test_enc = test_df[col].map(full_stats["smooth"]).fillna(global_mean).values

    return oof_enc, test_enc

# =========================
# 1) LOAD + ALIGN
# =========================
print("📂 Chargement...")
train = read_csv_robust(TRAIN_MERGED)
test  = read_csv_robust(TEST_MERGED)
y_df  = read_csv_robust(Y_PATH)

for df in (train, test, y_df):
    if "ID" not in df.columns:
        df.rename(columns={df.columns[0]: "ID"}, inplace=True)
    df["ID"] = normalize_id(df["ID"])

for df in (train, test):
    if "popularity" in df.columns:
        df.drop(columns=["popularity"], inplace=True)

train = train.merge(y_df[["ID","popularity"]], on="ID", how="left")
train = train.dropna(subset=["popularity"]).reset_index(drop=True)
print(f"   Train: {train.shape} | Test: {test.shape}")

all_feat = [c for c in train.columns if c not in ("ID","popularity")]
for c in set(all_feat) - set(test.columns):
    test[c] = np.nan
for c in set(test.columns) - set(all_feat) - {"ID"}:
    train[c] = np.nan
all_feat = sorted(set(train.columns) - {"ID","popularity"})

# =========================
# 2) FEATURE REDUCTION
# =========================
drop_redundant = set()
drop_redundant |= {f"mfcc_{i}_{s}" for i in range(6,20) for s in ("mean","std")}
drop_redundant |= {"width","height"}
for c in all_feat:
    if re.fullmatch(r".+_count", c) and c.replace("_count","_log_count") in all_feat:
        drop_redundant.add(c)
all_feat = [c for c in all_feat if c not in drop_redundant]

# =========================
# 3) TARGET + LOG TRANSFORM
# =========================
y_raw    = train["popularity"].astype(float).values
y_series = train["popularity"].astype(float)
y        = np.log1p(y_raw) if LOG_TARGET else y_raw
print(f"\n📐 Log-transform | skew original: {y_series.skew():.3f} → log: {pd.Series(y).skew():.3f}")

# =========================
# 4) TARGET ENCODING ✅
# =========================
print("\n🎯 Target Encoding (KFold + smoothing bayésien)...")

# --- 4a) Reconstituer la colonne uploader depuis les one-hot up_uploader_*
uploader_onehot = [c for c in all_feat if c.startswith("up_uploader_")]
if uploader_onehot:
    def reconstruct_cat(df, prefix_cols):
        """Reconstitue la catégorie depuis les colonnes one-hot."""
        cat_names = [c.replace("up_uploader_", "") for c in prefix_cols]
        mat = df[prefix_cols].fillna(0).values
        result = []
        for row in mat:
            idx = np.argmax(row)
            result.append(cat_names[idx] if row[idx] == 1 else "unknown")
        return pd.Series(result, index=df.index)

    train["_uploader_cat"] = reconstruct_cat(train, uploader_onehot)
    test["_uploader_cat"]  = reconstruct_cat(test,  uploader_onehot)
    uploader_te_cols = ["_uploader_cat"]
else:
    uploader_te_cols = []

# --- 4b) Colonnes continues discrétisées pour TE (ex: uploader_log_count binné)
# On binarize uploader_log_count en quintiles → proxy de "tier d'uploader"
if "uploader_log_count" in train.columns:
    train["_uploader_tier"] = pd.qcut(
        train["uploader_log_count"].fillna(0), q=5, labels=False, duplicates="drop"
    ).astype(str)
    test["_uploader_tier"] = pd.qcut(
        test["uploader_log_count"].fillna(0), q=5, labels=False, duplicates="drop"
    ).astype(str)
    tier_cols = ["_uploader_tier"]
else:
    tier_cols = []

# --- 4c) has_music × is_original_sound (interaction binaire)
if "has_music" in train.columns and "is_original_sound" in train.columns:
    train["_music_x_original"] = (
        train["has_music"].fillna(0).astype(str) + "_" +
        train["is_original_sound"].fillna(0).astype(str)
    )
    test["_music_x_original"] = (
        test["has_music"].fillna(0).astype(str) + "_" +
        test["is_original_sound"].fillna(0).astype(str)
    )
    music_cols = ["_music_x_original"]
else:
    music_cols = []

# --- 4d) Appliquer le target encoding sur toutes les colonnes cat
te_feature_names_train = {}
te_feature_names_test  = {}

all_te_cols = uploader_te_cols + tier_cols + music_cols
y_for_te    = pd.Series(y_raw)  # on encode sur y original (plus interprétable)

new_te_train = {}
new_te_test  = {}

for col in all_te_cols:
    if col in train.columns and col in test.columns:
        oof_enc, test_enc = target_encode_kfold(
            train, test, col, y_for_te, n_splits=N_SPLITS, smoothing=10
        )
        feat_name = f"te_{col}"
        new_te_train[feat_name] = oof_enc
        new_te_test[feat_name]  = test_enc
        print(f"   ✅ TE '{col}' → '{feat_name}'  (unique cats: {train[col].nunique()})")

te_train_df = pd.DataFrame(new_te_train, index=train.index)
te_test_df  = pd.DataFrame(new_te_test,  index=test.index)

TE_COLS = list(new_te_train.keys())
print(f"   → {len(TE_COLS)} nouvelles features TE : {TE_COLS}")

# =========================
# 5) BUILD FEATURE GROUPS
# =========================
onehot_cols = [c for c in all_feat if ONEHOT_PATTERN.match(c)]
color_xy    = [c for c in all_feat if re.match(r"^[RGBW]\d_[xy]$", c)]

GROUP2_VGG  = safe_cols(VGG_COLS,  train)
GROUP2_CLIP = safe_cols(CLIP_COLS, train)

# Groupe 1 : méta + librosa + TE ✅
G1_base  = safe_cols(META_COLS + LIBROSA_COLS + onehot_cols + color_xy, train)
G1_cols  = G1_base  # features standard
# On ajoutera les TE comme colonnes séparées via concat

print(f"\n📦 Groupes :")
print(f"   Groupe 1  (méta+librosa+TE)   : {len(G1_cols)} + {len(TE_COLS)} TE cols")
print(f"   Groupe 2  VGGish+CLIP (PCA)   : {len(GROUP2_VGG)}+{len(GROUP2_CLIP)} → {PCA_VGG_DIM+PCA_CLIP_DIM} dims")

# =========================
# 6) PCA EMBEDDINGS
# =========================
print("\n🔧 PCA embeddings...")
tr_vgg_pca,  te_vgg_pca  = apply_pca(train[GROUP2_VGG].values,  test[GROUP2_VGG].values,  PCA_VGG_DIM,  "vgg")
tr_clip_pca, te_clip_pca = apply_pca(train[GROUP2_CLIP].values, test[GROUP2_CLIP].values, PCA_CLIP_DIM, "clip")

# Feature matrices finales
def build_X(df, base_cols, te_df, pca_dfs=None):
    parts = [df[base_cols].fillna(0).reset_index(drop=True), te_df.reset_index(drop=True)]
    if pca_dfs:
        parts += [p.reset_index(drop=True) for p in pca_dfs]
    return pd.concat(parts, axis=1)

X1_tr = build_X(train, G1_cols, te_train_df)
X1_te = build_X(test,  G1_cols, te_test_df)

X2_tr = pd.concat([tr_vgg_pca, tr_clip_pca], axis=1)
X2_te = pd.concat([te_vgg_pca, te_clip_pca], axis=1)

# Groupe ALL : tout ensemble pour un modèle full
GALL_base = safe_cols(META_COLS + LIBROSA_COLS + VIDEO_COLS + onehot_cols + color_xy, train)
Xall_tr = build_X(train, GALL_base, te_train_df, [tr_vgg_pca, tr_clip_pca])
Xall_te = build_X(test,  GALL_base, te_test_df,  [te_vgg_pca, te_clip_pca])

print(f"\n   X1  (méta+librosa+TE)     : {X1_tr.shape[1]} features")
print(f"   X2  (VGGish+CLIP PCA)     : {X2_tr.shape[1]} features")
print(f"   Xall (tout+TE+PCA)        : {Xall_tr.shape[1]} features")

# =========================
# 7) LGB PARAMS
# =========================
LGB_PARAMS = {
    "objective":         "regression",
    "metric":            "rmse",
    "learning_rate":     0.05,
    "num_leaves":        31,
    "max_depth":         6,
    "min_child_samples": 30,
    "feature_fraction":  0.5,
    "bagging_fraction":  0.7,
    "bagging_freq":      5,
    "reg_alpha":         0.1,
    "reg_lambda":        1.0,
    "n_jobs":            -1,
    "verbose":           -1,
    "random_state":      SEED,
}

# =========================
# 8) KFOLD OOF
# =========================
def train_lgb_oof(X_tr, X_te, y, y_raw, name):
    kf              = KFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    oof_log         = np.zeros(len(X_tr))
    oof_orig        = np.zeros(len(X_tr))
    test_preds_log  = np.zeros(len(X_te))
    test_preds_orig = np.zeros(len(X_te))
    rmses = []

    print(f"\n{'─'*52}")
    print(f"  📊 {name}  |  {X_tr.shape[1]} features")
    print(f"{'─'*52}")

    for fold, (tr_idx, val_idx) in enumerate(kf.split(X_tr, y), 1):
        Xf_tr, Xf_val = X_tr.iloc[tr_idx], X_tr.iloc[val_idx]
        yf_tr, yf_val = y[tr_idx], y[val_idx]
        yf_val_raw    = y_raw[val_idx]

        model = lgb.LGBMRegressor(n_estimators=2000, **LGB_PARAMS)
        model.fit(
            Xf_tr, yf_tr,
            eval_set=[(Xf_val, yf_val)],
            callbacks=[
                lgb.early_stopping(100, verbose=False),
                lgb.log_evaluation(period=500),
            ]
        )
        pred_log = model.predict(Xf_val)
        pred_raw = np.expm1(pred_log) if LOG_TARGET else pred_log

        oof_log[val_idx]  = pred_log
        oof_orig[val_idx] = pred_raw

        tp_log           = model.predict(X_te)
        test_preds_log  += tp_log / N_SPLITS
        test_preds_orig += (np.expm1(tp_log) if LOG_TARGET else tp_log) / N_SPLITS

        fold_rmse = rmse(yf_val_raw, pred_raw)
        rmses.append(fold_rmse)
        print(f"    Fold {fold} | iter: {model.best_iteration_:4d} | RMSE: {fold_rmse:.4f}")

    oof_rmse = rmse(y_raw, oof_orig)
    print(f"  ✅ OOF RMSE {name}: {oof_rmse:.4f}")
    return oof_log, oof_orig, test_preds_log, test_preds_orig, oof_rmse

oof1_log,   oof1_orig,   te1_log,   te1_orig,   r1   = train_lgb_oof(X1_tr,   X1_te,   y, y_raw, "LGB-1  méta+librosa+TE")
oof2_log,   oof2_orig,   te2_log,   te2_orig,   r2   = train_lgb_oof(X2_tr,   X2_te,   y, y_raw, "LGB-2  VGGish+CLIP")
oofall_log, oofall_orig, teall_log, teall_orig, rall = train_lgb_oof(Xall_tr, Xall_te, y, y_raw, "LGB-ALL tout+TE+PCA")

# =========================
# 9) META-MODELE RIDGE
# =========================
print(f"\n{'='*52}")
print("  🧠 Meta-modèle Ridge")
print(f"{'='*52}")

meta_tr = np.column_stack([oof1_log,  oof2_log,  oofall_log])
meta_te = np.column_stack([te1_log,   te2_log,   teall_log])

meta_model = RidgeCV(alphas=[0.01, 0.1, 1.0, 10.0, 100.0], cv=5)
meta_model.fit(meta_tr, y)

print(f"  Alpha : {meta_model.alpha_}")
print(f"  Poids : LGB1={meta_model.coef_[0]:.3f} | LGB2={meta_model.coef_[1]:.3f} | LGBall={meta_model.coef_[2]:.3f}")

final_oof_log   = meta_model.predict(meta_tr)
final_preds_log = meta_model.predict(meta_te)
final_oof       = np.expm1(final_oof_log)   if LOG_TARGET else final_oof_log
final_preds     = np.expm1(final_preds_log) if LOG_TARGET else final_preds_log

stacking_rmse = rmse(y_raw, final_oof)

# =========================
# 10) BILAN
# =========================
print(f"\n{'='*52}")
print(f"  📊 BILAN FINAL")
print(f"{'='*52}")
print(f"  LGB-1  méta+librosa+TE  : {r1:.4f}")
print(f"  LGB-2  VGGish+CLIP      : {r2:.4f}")
print(f"  LGB-ALL tout+TE+PCA     : {rall:.4f}")
print(f"  ──────────────────────────────")
print(f"  🏆 STACKING             : {stacking_rmse:.4f}")
print(f"{'='*52}")

# =========================
# 11) SUBMISSION
# =========================
sub = pd.DataFrame({
    "ID":         test["ID"].astype(str).values,
    "popularity": final_preds,
})
assert sub["ID"].isna().sum() == 0
assert sub["popularity"].isna().sum() == 0

out_path = ROOT / "submission_stacking_v3_TE.csv"
sub.to_csv(out_path, index=False)
print(f"\n✅ Submission : {out_path.name}  |  shape: {sub.shape}")
print(f"   mean={final_preds.mean():.3f} | std={final_preds.std():.3f} | min={final_preds.min():.3f} | max={final_preds.max():.3f}")

📂 Chargement...
   Train: (1348, 951) | Test: (338, 950)

📐 Log-transform | skew original: 0.733 → log: 0.099

🎯 Target Encoding (KFold + smoothing bayésien)...
   ✅ TE '_uploader_cat' → 'te__uploader_cat'  (unique cats: 13)
   ✅ TE '_uploader_tier' → 'te__uploader_tier'  (unique cats: 4)
   ✅ TE '_music_x_original' → 'te__music_x_original'  (unique cats: 4)
   → 3 nouvelles features TE : ['te__uploader_cat', 'te__uploader_tier', 'te__music_x_original']

📦 Groupes :
   Groupe 1  (méta+librosa+TE)   : 99 + 3 TE cols
   Groupe 2  VGGish+CLIP (PCA)   : 256+512 → 128 dims

🔧 PCA embeddings...
   PCA vgg: 256 → 64 dims | var: 84.1%
   PCA clip: 512 → 64 dims | var: 63.0%

   X1  (méta+librosa+TE)     : 102 features
   X2  (VGGish+CLIP PCA)     : 128 features
   Xall (tout+TE+PCA)        : 289 features

────────────────────────────────────────────────────
  📊 LGB-1  méta+librosa+TE  |  102 features
────────────────────────────────────────────────────
    Fold 1 | iter:   71 | RMSE: 1.2359
  